# 3 Baseline Multi-Label Classifier

As a baseline, we trained a simple multi-label text classifier using TF-IDF features and silver labels generated from keyword matching. The model is a feed-forward neural network with sigmoid outputs and binary cross-entropy loss, optimized using Adam and early stopping based on validation micro-F1. This baseline allows us to assess whether the automatically generated silver labels contain meaningful signal and establishes a reference point for future improvements such as pseudo-labeling or hierarchy-aware training.

In [5]:
# Relevant imports
import os
import csv
import copy
import random
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path
import numpy as np
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Setup initial filepaths
ROOT = Path("project_release")

TRAIN_CORPUS_PATH = ROOT / "Amazon_products" / "train" / "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "Amazon_products" / "test" / "test_corpus.txt"
SILVER_PATH = ROOT / "silver_labels"

SUBMISSION_PATH = ROOT / "submissions"
SUBMISSION_PATH.mkdir(exist_ok=True)

import os


In [6]:
"""
# TF-IDF silver labels
with open(SILVER_PATH / "silver_labels_tfidf.pkl", "rb") as f:
    silver_labels_tfidf = pickle.load(f)
"""
# SVD silver labels
with open(SILVER_PATH / "silver_labels_svd.pkl", "rb") as f:
    silver_labels_svd = pickle.load(f)

# BERT silver labels
with open(SILVER_PATH / "silver_labels_bert.pkl", "rb") as f:
    silver_labels_bert = pickle.load(f)

# Hybrid silver labels
with open(SILVER_PATH / "silver_labels_hybrid.pkl", "rb") as f:
    hybrid_labels = pickle.load(f)

In [7]:
def silver_to_dict(silver_labels):
    """
    Converts silver label list [(pid, [(class_name, score), ...]), ...] 
    to {pid: [class_name1, class_name2, ...]} dictionary
    """
    review_to_classes = defaultdict(list)
    for pid, cls_list in silver_labels:
        review_to_classes[pid] = [cls for cls, score in cls_list]
    return review_to_classes

# Example for hybrid labels
review_to_classes_hybrid = silver_to_dict(hybrid_labels)

In [8]:
class ReviewsDataset(Dataset):
    def __init__(self, corpus_path, review_to_classes, vectorizer=None, fit_vectorizer=False, max_features=5000):
        self.texts = []
        self.labels = []
        self.review_ids = []
        
        # Collect all unique classes
        self.all_classes = sorted({cls for classes in review_to_classes.values() for cls in classes})
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.all_classes)}
        
        # Read corpus
        with open(corpus_path, "r", encoding="utf8") as f:
            for idx, line in enumerate(f):
                text = line.strip()
                self.texts.append(text)
                self.review_ids.append(idx)
                
                classes = review_to_classes.get(idx, [])
                label = [0] * len(self.all_classes)
                for cls in classes:
                    if cls in self.class_to_idx:
                        label[self.class_to_idx[cls]] = 1
                self.labels.append(label)
        
        # Vectorization
        if vectorizer is None:
            self.vectorizer = TfidfVectorizer(max_features=max_features)
        else:
            self.vectorizer = vectorizer
        
        if fit_vectorizer:
            self.features = self.vectorizer.fit_transform(self.texts).toarray()
        else:
            self.features = self.vectorizer.transform(self.texts).toarray()
        
        self.labels = torch.tensor(self.labels, dtype=torch.float32)
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return {"X": torch.tensor(self.features[idx], dtype=torch.float32),
                "y": self.labels[idx]}


In [9]:
# Example using hybrid labels
dataset = ReviewsDataset(TRAIN_CORPUS_PATH, review_to_classes_hybrid, fit_vectorizer=True)
vectorizer = dataset.vectorizer

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)



In [10]:
class MultiLabelClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.out = nn.Linear(256, output_dim)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.out(x)
        return torch.sigmoid(x)


In [11]:
input_dim = dataset.features.shape[1]
output_dim = len(dataset.all_classes)

model = MultiLabelClassifier(input_dim, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

EPOCHS = 100
patience = 5
best_val_f1 = -1
patience_counter = 0
best_model_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        X, y = batch["X"].to(device), batch["y"].to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"[Epoch {epoch}] Train Loss: {avg_loss:.4f}")
    
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            X, y = batch["X"].to(device), batch["y"].to(device)
            preds = (model(X) > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()
    val_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    print(f"[VAL] f1_micro: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print("[Early Stopping] No improvement.")
        break

# Save the best model
torch.save(best_model_state, SILVER_PATH / "model_hybrid.pth")


[Epoch 1] Train Loss: 0.0423
[VAL] f1_micro: 0.5537
[Epoch 2] Train Loss: 0.0183
[VAL] f1_micro: 0.6297
[Epoch 3] Train Loss: 0.0153
[VAL] f1_micro: 0.6738
[Epoch 4] Train Loss: 0.0134
[VAL] f1_micro: 0.6914
[Epoch 5] Train Loss: 0.0119
[VAL] f1_micro: 0.7017
[Epoch 6] Train Loss: 0.0107
[VAL] f1_micro: 0.7105
[Epoch 7] Train Loss: 0.0095
[VAL] f1_micro: 0.7121
[Epoch 8] Train Loss: 0.0085
[VAL] f1_micro: 0.7146
[Epoch 9] Train Loss: 0.0075
[VAL] f1_micro: 0.7144
[Epoch 10] Train Loss: 0.0067
[VAL] f1_micro: 0.7123
[Epoch 11] Train Loss: 0.0060
[VAL] f1_micro: 0.7104
[Epoch 12] Train Loss: 0.0053
[VAL] f1_micro: 0.7088
[Epoch 13] Train Loss: 0.0048
[VAL] f1_micro: 0.7118
[Early Stopping] No improvement.


KAGGLE SUBMISSION

In [ ]:
import csv
import torch

def generate_submission(model, vectorizer, test_corpus_path, submission_path, device="cpu", threshold=0.5, batch_size=64, min_labels=1):
    """
    Generate multi-label predictions and save to CSV for Kaggle submission.

    Args:
        model: trained PyTorch model
        vectorizer: fitted TF-IDF vectorizer
        test_corpus_path: path to test corpus (tab-separated: id \t text)
        submission_path: path to output CSV
        device: "cpu" or "cuda"
        threshold: probability threshold for multi-label classification
        batch_size: number of samples per batch
        min_labels: minimum number of labels per sample
    """
    # Load test corpus
    pid2text = {}
    with open(test_corpus_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                pid, text = parts
                pid2text[pid] = text

    pid_list = list(pid2text.keys())
    test_texts = [pid2text[pid] for pid in pid_list]
    
    # Transform features
    X_test = vectorizer.transform(test_texts).toarray()
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Predict
    model.eval()
    all_labels = []
    with torch.no_grad():
        for i in range(0, len(X_test), batch_size):
            X_batch = X_test[i:i+batch_size]
            preds = model(X_batch)
            preds = (preds > threshold).cpu().numpy()

            for row in preds:
                labels = list(row.nonzero()[0])
                if len(labels) < min_labels:
                    labels = [0]  # fallback to default class
                all_labels.append(labels)

    # Save to CSV
    with open(submission_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["pid", "labels"])
        for pid, labels in zip(pid_list, all_labels):
            writer.writerow([pid, ",".join(map(str, labels))])

    # Print stats
    num_labels_per_sample = [len(lbls) for lbls in all_labels]
    print(f"Predicted classes per sample: min={min(num_labels_per_sample)}, "
          f"max={max(num_labels_per_sample)}, avg={sum(num_labels_per_sample)/len(num_labels_per_sample):.2f}")

    return all_labels

def count_empty_predictions(all_labels, pid_list=None, verbose=False):
    """
    Count and optionally print predictions that are empty or None.
    """
    empty_count = 0
    for idx, labels in enumerate(all_labels):
        if labels is None or len(labels) == 0:
            empty_count += 1
            if verbose and pid_list is not None:
                print(f"Empty prediction at index {idx}, PID: {pid_list[idx]}")
    total = len(all_labels)
    print(f"\nTotal empty predictions: {empty_count} out of {total} ({empty_count/total:.2%})")
    return empty_count



In [19]:
submission_file = SUBMISSION_PATH / "submission.csv"

all_labels = generate_submission(
    model=model,
    vectorizer=vectorizer,
    test_corpus_path=TEST_CORPUS_PATH,
    submission_path=submission_file,  # CSV file
    device=device,
    threshold=0.5
)


Predicted classes per sample: min=1, max=6, avg=2.72


In [14]:
def count_empty_predictions(predictions, pid_list=None, verbose=True):
    count = 0
    for i, pred in enumerate(predictions):
        if not pred:  # empty list or None
            count += 1
            if verbose:
                print(f"Empty prediction for pid {pid_list[i]}")
    return count

all_labels_list = [all_labels[pid] for pid in pid_list_test]
empty_count = count_empty_predictions(all_labels_list, pid_list=pid_list_test, verbose=False)


NameError: name 'pid_list_test' is not defined

In [15]:
# Count how many predictions were forced to the fallback [0]
defaulted = sum(1 for lbl in all_labels if lbl == [0])
total = len(all_labels)

print(f"Defaulted predictions: {defaulted}/{total} "
      f"({defaulted / total:.2%})")


Defaulted predictions: 52/19658 (0.26%)
